# Convert and upload your GRPO checkpoint

Use the same Nebius kernel/storage as training, or use a local kernel after downloading your checkpoint. No repository checkout is required. Export helpers are embedded in this notebook. Conversion runs on CPU in a separate process.

Create your token at https://huggingface.co/settings/tokens with **Write** permission (or fine-grained write access to the destination model repository). Enter it only into the hidden prompt below. The notebook does not store the token in its cells or save it to disk.

Run conversion and validation first. Only the final cell uploads. Repositories are private by default.


In [1]:
import sys
import subprocess
import importlib.util
from pathlib import Path

packages = ["transformers", "accelerate", "huggingface_hub", "safetensors"]
if importlib.util.find_spec("reasoning_from_scratch") is None:
    packages.append("git+https://github.com/RRisto/reasoning-from-scratch.git")
subprocess.run([sys.executable, "-m", "pip", "install", *packages], check=True)


CompletedProcess(args=['/opt/conda/bin/python', '-m', 'pip', 'install', 'transformers', 'accelerate', 'huggingface_hub', 'safetensors'], returncode=0)

## Select a checkpoint and destination
Use step 50 or 100 based on your measured evaluation, not the example name alone. Paths refer to the selected kernel filesystem. Set CHECKPOINT explicitly if automatic folder detection misses your files.

In [2]:
TRAINING_DIR = Path.cwd() / "grpo_training"
CHECKPOINT_DIR = TRAINING_DIR / "checkpoints" / "rlvr_grpo_original_no_kl"
print("Available checkpoints:")
for path in sorted(CHECKPOINT_DIR.glob("*.pth")):
    print(path)

CHECKPOINT = CHECKPOINT_DIR / "qwen3-0.6B-rlvr-grpo-step00100.pth"
EXPORT_DIR = Path.cwd() / "qwen3-grpo-hf"
REPO_ID = "YOUR_USERNAME/qwen3-0.6b-grpo-math"
PRIVATE = True
# Actual training configuration; edit to match your run.
TRAINING_NOTES = "GRPO without KL; 8 rollouts; maximum 1024 generated tokens; checkpoint step 100."
EVALUATION_NOTES = "Step 100: 28/50 correct (56%) on the 50-example MATH-500 subset, with a 1024-token generation limit. Step 50: 21/50 (42%). These subset scores are not a full MATH-500 benchmark."


Available checkpoints:
/home/jovyan/grpo_training/checkpoints/rlvr_grpo_original_no_kl/qwen3-0.6B-rlvr-grpo-step00050.pth
/home/jovyan/grpo_training/checkpoints/rlvr_grpo_original_no_kl/qwen3-0.6B-rlvr-grpo-step00100.pth
/home/jovyan/grpo_training/checkpoints/rlvr_grpo_original_no_kl/qwen3-0.6B-rlvr-grpo.pth


## Prepare the conversion helpers
Embedded copies of this repository's exporter and custom Hugging Face architecture.

In [3]:
EXPORT_HELPERS = {'hf_export.py': '# Copyright (c) Sebastian Raschka under Apache License 2.0 (see LICENSE.txt)\n# Source for "Build a Reasoning Model (From Scratch)": https://mng.bz/lZ5B\n# Code repository: https://github.com/rasbt/reasoning-from-scratch\n\nimport argparse\nimport json\nfrom pathlib import Path\n\nimport torch\nfrom transformers import GenerationConfig, PreTrainedTokenizerFast\n\nfrom hf_qwen3 import (\n    Qwen3FromScratchConfig,\n    Qwen3FromScratchForCausalLM,\n)\nfrom reasoning_from_scratch.qwen3 import QWEN_CONFIG_06_B, download_qwen3_small\n\n\nMODEL_FILENAMES = {\n    "base": "qwen3-0.6B-base.pth",\n    "reasoning": "qwen3-0.6B-reasoning.pth",\n}\n\nTOKENIZER_FILENAMES = {\n    "base": "tokenizer-base.json",\n    "reasoning": "tokenizer-reasoning.json",\n}\n\nEOS_TOKENS = {\n    "base": "<|endoftext|>",\n    "reasoning": "<|im_end|>",\n}\n\nPAD_TOKEN = "<|endoftext|>"\n\nREASONING_CHAT_TEMPLATE = """{% for message in messages %}<|im_start|>{{ message[\'role\'] }}\n{{ message[\'content\'] }}<|im_end|>\n{% endfor %}{% if add_generation_prompt %}<|im_start|>assistant\n{% endif %}"""\n\n\ndef torch_dtype_to_name(dtype_value):\n    # The scratch config stores dtypes as torch objects such as torch.bfloat16,\n    # but the Hugging Face config is saved as JSON, so we convert them to strings\n    # like "bfloat16" before writing config.json.\n    if isinstance(dtype_value, torch.dtype):\n        return str(dtype_value).removeprefix("torch.")\n    if isinstance(dtype_value, str):\n        return dtype_value.removeprefix("torch.")\n    raise TypeError(f"Unsupported dtype value: {dtype_value!r}")\n\n\ndef resolve_source_paths(\n    tokenizer_kind="base",\n    model_path=None,\n    tokenizer_path=None,\n    download_dir="qwen3",\n):\n    download_dir = Path(download_dir)\n\n    if model_path is not None:\n        model_path = Path(model_path)\n    else:\n        download_qwen3_small(\n            kind=tokenizer_kind,\n            tokenizer_only=False,\n            out_dir=download_dir,\n        )\n        model_path = download_dir / MODEL_FILENAMES[tokenizer_kind]\n\n    if tokenizer_path is not None:\n        tokenizer_path = Path(tokenizer_path)\n    else:\n        download_qwen3_small(\n            kind=tokenizer_kind,\n            tokenizer_only=True,\n            out_dir=download_dir,\n        )\n        tokenizer_path = download_dir / TOKENIZER_FILENAMES[tokenizer_kind]\n\n    return model_path, tokenizer_path\n\n\ndef build_hf_tokenizer(tokenizer_path, tokenizer_kind, model_max_length):\n    tokenizer = PreTrainedTokenizerFast(\n        tokenizer_file=str(tokenizer_path),\n        pad_token=PAD_TOKEN,\n        eos_token=EOS_TOKENS[tokenizer_kind],\n    )\n    tokenizer.model_max_length = model_max_length\n    tokenizer.padding_side = "right"\n    tokenizer.truncation_side = "right"\n\n    if tokenizer_kind == "reasoning":\n        tokenizer.chat_template = REASONING_CHAT_TEMPLATE\n\n    return tokenizer\n\n\ndef build_hf_config(tokenizer, tokenizer_kind, model_cfg=None):\n    if model_cfg is None:\n        model_cfg = dict(QWEN_CONFIG_06_B)\n\n    return Qwen3FromScratchConfig(\n        vocab_size=model_cfg["vocab_size"],\n        hidden_size=model_cfg["emb_dim"],\n        num_hidden_layers=model_cfg["n_layers"],\n        num_attention_heads=model_cfg["n_heads"],\n        intermediate_size=model_cfg["hidden_dim"],\n        num_key_value_heads=model_cfg["n_kv_groups"],\n        head_dim=model_cfg["head_dim"],\n        max_position_embeddings=model_cfg["context_length"],\n        rope_theta=model_cfg["rope_base"],\n        use_qk_norm=model_cfg["qk_norm"],\n        dtype=torch_dtype_to_name(model_cfg["dtype"]),\n        tokenizer_kind=tokenizer_kind,\n        pad_token_id=tokenizer.pad_token_id,\n        eos_token_id=tokenizer.eos_token_id,\n    )\n\n\ndef export_qwen3_to_hf(\n    output_dir,\n    state_dict=None,\n    tokenizer_path=None,\n    tokenizer_kind="base",\n    model_path=None,\n    download_dir="qwen3",\n    model_cfg=None,\n    source_path=None,\n):\n    if state_dict is not None and model_path is not None:\n        raise ValueError("Pass either state_dict or model_path, not both.")\n\n    if state_dict is None:\n        source_path, tokenizer_path = resolve_source_paths(\n            tokenizer_kind=tokenizer_kind,\n            model_path=model_path,\n            tokenizer_path=tokenizer_path,\n            download_dir=download_dir,\n        )\n        try:\n            state_dict = torch.load(source_path, map_location="cpu", weights_only=True)\n        except TypeError:\n            state_dict = torch.load(source_path, map_location="cpu")\n    else:\n        if tokenizer_path is None:\n            download_dir = Path(download_dir)\n            download_qwen3_small(\n                kind=tokenizer_kind,\n                tokenizer_only=True,\n                out_dir=download_dir,\n            )\n            tokenizer_path = download_dir / TOKENIZER_FILENAMES[tokenizer_kind]\n        else:\n            tokenizer_path = Path(tokenizer_path)\n\n    output_dir = Path(output_dir)\n    output_dir.mkdir(parents=True, exist_ok=True)\n\n    tokenizer = build_hf_tokenizer(\n        tokenizer_path=tokenizer_path,\n        tokenizer_kind=tokenizer_kind,\n        model_max_length=(\n            model_cfg["context_length"]\n            if model_cfg is not None\n            else QWEN_CONFIG_06_B["context_length"]\n        ),\n    )\n    config = build_hf_config(\n        tokenizer=tokenizer,\n        tokenizer_kind=tokenizer_kind,\n        model_cfg=model_cfg,\n    )\n\n    Qwen3FromScratchConfig.register_for_auto_class("AutoConfig")\n    Qwen3FromScratchForCausalLM.register_for_auto_class("AutoModelForCausalLM")\n\n    model = Qwen3FromScratchForCausalLM(config)\n    state_dict = dict(state_dict)\n    state_dict.setdefault("cos", model.cos)\n    state_dict.setdefault("sin", model.sin)\n    missing_keys, unexpected_keys = model.load_state_dict(state_dict, strict=True)\n    if missing_keys or unexpected_keys:\n        raise RuntimeError(\n            "State dict mismatch during export. "\n            f"Missing keys: {missing_keys}. Unexpected keys: {unexpected_keys}."\n        )\n\n    model.save_pretrained(\n        output_dir,\n        safe_serialization=True,\n    )\n    tokenizer.save_pretrained(output_dir)\n\n    generation_config = GenerationConfig(\n        eos_token_id=tokenizer.eos_token_id,\n        pad_token_id=tokenizer.pad_token_id,\n    )\n    generation_config.save_pretrained(output_dir)\n\n    export_info = {\n        "tokenizer_kind": tokenizer_kind,\n        "source_path": str(source_path) if source_path is not None else None,\n        "safe_serialization": True,\n    }\n    (output_dir / "export_info.json").write_text(\n        json.dumps(export_info, indent=2),\n        encoding="utf-8",\n    )\n\n    return output_dir\ndef parse_args():\n    parser = argparse.ArgumentParser(\n        description=(\n            "Convert the scratch Qwen3 model or a saved .pth checkpoint into "\n            "a Hugging Face Transformers-compatible folder."\n        ),\n        formatter_class=argparse.ArgumentDefaultsHelpFormatter,\n    )\n    parser.add_argument(\n        "--output_dir",\n        type=str,\n        required=True,\n        help="Directory where the Hugging Face model folder will be written.",\n    )\n    parser.add_argument(\n        "--tokenizer_kind",\n        type=str,\n        choices=("base", "reasoning"),\n        default="base",\n        help=(\n            "Tokenizer family to bundle with the export. For chapter 8 "\n            "distillation checkpoints, use \'reasoning\'."\n        ),\n    )\n    parser.add_argument(\n        "--model_path",\n        type=str,\n        default=None,\n        help=(\n            "Optional local .pth file to export. This can be either the "\n            "original Qwen3 weights or a trained checkpoint."\n        ),\n    )\n    parser.add_argument(\n        "--tokenizer_path",\n        type=str,\n        default=None,\n        help=(\n            "Optional local path to tokenizer-base.json or "\n            "tokenizer-reasoning.json. If omitted, the tokenizer is downloaded."\n        ),\n    )\n    parser.add_argument(\n        "--download_dir",\n        type=str,\n        default="qwen3",\n        help="Cache directory used when the script downloads source weights/tokenizers.",\n    )\n    return parser.parse_args()\n\n\ndef main():\n    args = parse_args()\n\n    export_dir = export_qwen3_to_hf(\n        output_dir=args.output_dir,\n        tokenizer_kind=args.tokenizer_kind,\n        model_path=args.model_path,\n        tokenizer_path=args.tokenizer_path,\n        download_dir=args.download_dir,\n    )\n\n    print(f"Exported Hugging Face model folder to {export_dir}")\n\n\nif __name__ == "__main__":\n    main()\n', 'hf_qwen3.py': '# Copyright (c) Sebastian Raschka under Apache License 2.0 (see LICENSE.txt)\n# Source for "Build a Reasoning Model (From Scratch)": https://mng.bz/lZ5B\n# Code repository: https://github.com/rasbt/reasoning-from-scratch\n\nfrom typing import Optional\n\nimport torch\nimport torch.nn as nn\nfrom transformers import GenerationMixin, PreTrainedModel, PretrainedConfig\nfrom transformers.modeling_outputs import CausalLMOutputWithPast\n\nfrom reasoning_from_scratch.qwen3 import RMSNorm, apply_rope, compute_rope_params\n\n\nclass Qwen3FromScratchConfig(PretrainedConfig):\n    model_type = "qwen3_from_scratch"\n    keys_to_ignore_at_inference = ["past_key_values"]\n\n    def __init__(\n        self,\n        vocab_size=151_936,\n        hidden_size=1024,\n        num_hidden_layers=28,\n        num_attention_heads=16,\n        intermediate_size=3072,\n        num_key_value_heads=8,\n        head_dim=128,\n        max_position_embeddings=40_960,\n        rope_theta=1_000_000.0,\n        rms_norm_eps=1e-6,\n        use_qk_norm=True,\n        tokenizer_kind="base",\n        pad_token_id=151_643,\n        eos_token_id=151_643,\n        bos_token_id=None,\n        dtype="bfloat16",\n        torch_dtype=None,\n        initializer_range=0.02,\n        emb_dim=None,\n        n_layers=None,\n        n_heads=None,\n        hidden_dim=None,\n        n_kv_groups=None,\n        context_length=None,\n        rope_base=None,\n        qk_norm=None,\n        **kwargs,\n    ):\n        if emb_dim is not None:\n            hidden_size = emb_dim\n        if n_layers is not None:\n            num_hidden_layers = n_layers\n        if n_heads is not None:\n            num_attention_heads = n_heads\n        if hidden_dim is not None:\n            intermediate_size = hidden_dim\n        if n_kv_groups is not None:\n            num_key_value_heads = n_kv_groups\n        if context_length is not None:\n            max_position_embeddings = context_length\n        if rope_base is not None:\n            rope_theta = rope_base\n        if qk_norm is not None:\n            use_qk_norm = qk_norm\n        if torch_dtype is not None:\n            dtype = torch_dtype\n\n        self.use_cache = kwargs.pop("use_cache", True)\n        self.vocab_size = vocab_size\n        self.hidden_size = hidden_size\n        self.num_hidden_layers = num_hidden_layers\n        self.num_attention_heads = num_attention_heads\n        self.intermediate_size = intermediate_size\n        self.num_key_value_heads = num_key_value_heads\n        self.head_dim = head_dim\n        self.max_position_embeddings = max_position_embeddings\n        self.rope_theta = rope_theta\n        self.rms_norm_eps = rms_norm_eps\n        self.use_qk_norm = use_qk_norm\n        self.tokenizer_kind = tokenizer_kind\n        self.initializer_range = initializer_range\n\n        self.emb_dim = hidden_size\n        self.n_layers = num_hidden_layers\n        self.n_heads = num_attention_heads\n        self.hidden_dim = intermediate_size\n        self.n_kv_groups = num_key_value_heads\n        self.context_length = max_position_embeddings\n        self.rope_base = rope_theta\n        self.qk_norm = use_qk_norm\n\n        kwargs.pop("tie_word_embeddings", None)\n\n        super().__init__(\n            pad_token_id=pad_token_id,\n            eos_token_id=eos_token_id,\n            bos_token_id=bos_token_id,\n            dtype=dtype,\n            tie_word_embeddings=False,\n            **kwargs,\n        )\n\n\ndef _resolve_torch_dtype(dtype_value):\n    if isinstance(dtype_value, torch.dtype):\n        return dtype_value\n    if dtype_value is None:\n        return torch.float32\n    if isinstance(dtype_value, str):\n        return getattr(torch, dtype_value)\n    raise TypeError(f"Unsupported torch dtype value: {dtype_value!r}")\n\n\nclass FeedForward(nn.Module):\n    def __init__(self, config, dtype):\n        super().__init__()\n        self.fc1 = nn.Linear(\n            config.hidden_size,\n            config.intermediate_size,\n            bias=False,\n            dtype=dtype,\n        )\n        self.fc2 = nn.Linear(\n            config.hidden_size,\n            config.intermediate_size,\n            bias=False,\n            dtype=dtype,\n        )\n        self.fc3 = nn.Linear(\n            config.intermediate_size,\n            config.hidden_size,\n            bias=False,\n            dtype=dtype,\n        )\n\n    def forward(self, x):\n        x_fc1 = self.fc1(x)\n        x_fc2 = self.fc2(x)\n        x = nn.functional.silu(x_fc1) * x_fc2\n        return self.fc3(x)\n\n\nclass GroupedQueryAttention(nn.Module):\n    def __init__(self, config, dtype):\n        super().__init__()\n        assert (\n            config.num_attention_heads % config.num_key_value_heads == 0\n        ), "num_attention_heads must be divisible by num_key_value_heads"\n\n        self.num_heads = config.num_attention_heads\n        self.num_kv_groups = config.num_key_value_heads\n        self.group_size = self.num_heads // self.num_kv_groups\n        self.head_dim = config.head_dim\n        self.d_out = self.num_heads * self.head_dim\n\n        self.W_query = nn.Linear(\n            config.hidden_size,\n            self.d_out,\n            bias=False,\n            dtype=dtype,\n        )\n        self.W_key = nn.Linear(\n            config.hidden_size,\n            self.num_kv_groups * self.head_dim,\n            bias=False,\n            dtype=dtype,\n        )\n        self.W_value = nn.Linear(\n            config.hidden_size,\n            self.num_kv_groups * self.head_dim,\n            bias=False,\n            dtype=dtype,\n        )\n        self.out_proj = nn.Linear(\n            self.d_out,\n            config.hidden_size,\n            bias=False,\n            dtype=dtype,\n        )\n\n        if config.use_qk_norm:\n            self.q_norm = RMSNorm(self.head_dim, eps=config.rms_norm_eps)\n            self.k_norm = RMSNorm(self.head_dim, eps=config.rms_norm_eps)\n        else:\n            self.q_norm = None\n            self.k_norm = None\n\n    def forward(self, x, mask, cos, sin, start_pos=0, cache=None, layer_idx=None):\n        batch_size, num_tokens, _ = x.shape\n\n        queries = self.W_query(x)\n        keys = self.W_key(x)\n        values = self.W_value(x)\n\n        queries = queries.view(batch_size, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)\n        keys_new = keys.view(batch_size, num_tokens, self.num_kv_groups, self.head_dim).transpose(1, 2)\n        values_new = values.view(batch_size, num_tokens, self.num_kv_groups, self.head_dim).transpose(1, 2)\n\n        if self.q_norm is not None:\n            queries = self.q_norm(queries)\n        if self.k_norm is not None:\n            keys_new = self.k_norm(keys_new)\n\n        queries = apply_rope(queries, cos, sin, offset=start_pos)\n        keys_new = apply_rope(keys_new, cos, sin, offset=start_pos)\n\n        if hasattr(cache, "update") and layer_idx is not None:\n            keys, values = cache.update(keys_new, values_new, layer_idx)\n        elif cache is not None:\n            prev_k, prev_v = cache\n            keys = torch.cat([prev_k, keys_new], dim=2)\n            values = torch.cat([prev_v, values_new], dim=2)\n        else:\n            keys, values = keys_new, values_new\n\n        next_cache = (keys, values)\n        keys = keys.repeat_interleave(self.group_size, dim=1)\n        values = values.repeat_interleave(self.group_size, dim=1)\n\n        attn_scores = queries @ keys.transpose(2, 3)\n        attn_scores = attn_scores.masked_fill(mask, -torch.inf)\n        attn_weights = torch.softmax(attn_scores / self.head_dim**0.5, dim=-1)\n\n        context = (attn_weights @ values).transpose(1, 2).reshape(\n            batch_size,\n            num_tokens,\n            self.d_out,\n        )\n        return self.out_proj(context), next_cache\n\n\nclass TransformerBlock(nn.Module):\n    def __init__(self, config, dtype):\n        super().__init__()\n        self.att = GroupedQueryAttention(config, dtype=dtype)\n        self.ff = FeedForward(config, dtype=dtype)\n        self.norm1 = RMSNorm(config.hidden_size, eps=config.rms_norm_eps)\n        self.norm2 = RMSNorm(config.hidden_size, eps=config.rms_norm_eps)\n\n    def forward(self, x, mask, cos, sin, start_pos=0, cache=None, layer_idx=None):\n        shortcut = x\n        x = self.norm1(x)\n        x, next_cache = self.att(\n            x,\n            mask,\n            cos,\n            sin,\n            start_pos=start_pos,\n            cache=cache,\n            layer_idx=layer_idx,\n        )\n        x = x + shortcut\n\n        shortcut = x\n        x = self.norm2(x)\n        x = self.ff(x)\n        x = x + shortcut\n\n        return x, next_cache\n\n\nclass Qwen3FromScratchForCausalLM(PreTrainedModel, GenerationMixin):\n    config_class = Qwen3FromScratchConfig\n    main_input_name = "input_ids"\n\n    def __init__(self, config):\n        super().__init__(config)\n        self.param_dtype = _resolve_torch_dtype(config.dtype)\n\n        self.tok_emb = nn.Embedding(\n            config.vocab_size,\n            config.hidden_size,\n            dtype=self.param_dtype,\n        )\n        self.trf_blocks = nn.ModuleList(\n            [\n                TransformerBlock(config, dtype=self.param_dtype)\n                for _ in range(config.num_hidden_layers)\n            ]\n        )\n        self.final_norm = RMSNorm(config.hidden_size, eps=config.rms_norm_eps)\n        self.out_head = nn.Linear(\n            config.hidden_size,\n            config.vocab_size,\n            bias=False,\n            dtype=self.param_dtype,\n        )\n\n        cos, sin = compute_rope_params(\n            head_dim=config.head_dim,\n            theta_base=config.rope_theta,\n            context_length=config.max_position_embeddings,\n        )\n        self.register_buffer("cos", cos)\n        self.register_buffer("sin", sin)\n\n        self.post_init()\n\n    def _init_weights(self, module):\n        if isinstance(module, nn.Linear):\n            nn.init.normal_(module.weight, mean=0.0, std=self.config.initializer_range)\n            if module.bias is not None:\n                nn.init.zeros_(module.bias)\n        elif isinstance(module, nn.Embedding):\n            nn.init.normal_(module.weight, mean=0.0, std=self.config.initializer_range)\n        elif isinstance(module, RMSNorm):\n            nn.init.ones_(module.scale)\n            if module.shift is not None:\n                nn.init.zeros_(module.shift)\n\n    def get_input_embeddings(self):\n        return self.tok_emb\n\n    def set_input_embeddings(self, value):\n        self.tok_emb = value\n\n    def get_output_embeddings(self):\n        return self.out_head\n\n    def set_output_embeddings(self, new_embeddings):\n        self.out_head = new_embeddings\n\n    def _build_causal_mask(self, attention_mask, batch_size, seq_len, past_len, device):\n        total_kv_len = past_len + seq_len\n        query_positions = torch.arange(\n            past_len,\n            past_len + seq_len,\n            device=device,\n        ).unsqueeze(1)\n        key_positions = torch.arange(total_kv_len, device=device).unsqueeze(0)\n        causal_mask = key_positions > query_positions\n        causal_mask = causal_mask.unsqueeze(0).unsqueeze(0)\n\n        if attention_mask is None:\n            return causal_mask\n\n        if attention_mask.size(1) < total_kv_len:\n            prefix = torch.ones(\n                batch_size,\n                total_kv_len - attention_mask.size(1),\n                dtype=attention_mask.dtype,\n                device=attention_mask.device,\n            )\n            attention_mask = torch.cat([prefix, attention_mask], dim=1)\n        elif attention_mask.size(1) > total_kv_len:\n            attention_mask = attention_mask[:, -total_kv_len:]\n\n        key_padding_mask = attention_mask[:, None, None, :].eq(0)\n        return causal_mask | key_padding_mask\n\n    def forward(\n        self,\n        input_ids: Optional[torch.LongTensor] = None,\n        attention_mask: Optional[torch.Tensor] = None,\n        labels: Optional[torch.LongTensor] = None,\n        past_key_values=None,\n        use_cache: Optional[bool] = None,\n        return_dict: Optional[bool] = None,\n        **kwargs,\n    ):\n        if input_ids is None:\n            raise ValueError("input_ids must be provided.")\n\n        use_cache = self.config.use_cache if use_cache is None else use_cache\n        return_dict = self.config.use_return_dict if return_dict is None else return_dict\n        is_dynamic_cache = hasattr(past_key_values, "update") and hasattr(\n            past_key_values,\n            "get_seq_length",\n        )\n\n        hidden_states = self.tok_emb(input_ids)\n\n        batch_size, seq_len = input_ids.shape\n        past_len = 0\n        if is_dynamic_cache:\n            past_len = past_key_values.get_seq_length(0)\n        elif past_key_values:\n            past_len = past_key_values[0][0].shape[2]\n\n        mask = self._build_causal_mask(\n            attention_mask=attention_mask,\n            batch_size=batch_size,\n            seq_len=seq_len,\n            past_len=past_len,\n            device=input_ids.device,\n        )\n\n        next_past_key_values = [] if use_cache else None\n        for layer_idx, block in enumerate(self.trf_blocks):\n            if is_dynamic_cache:\n                hidden_states, next_cache = block(\n                    hidden_states,\n                    mask,\n                    self.cos,\n                    self.sin,\n                    start_pos=past_len,\n                    cache=past_key_values,\n                    layer_idx=layer_idx,\n                )\n            else:\n                layer_past = past_key_values[layer_idx] if past_key_values else None\n                hidden_states, next_cache = block(\n                    hidden_states,\n                    mask,\n                    self.cos,\n                    self.sin,\n                    start_pos=past_len,\n                    cache=layer_past,\n                    layer_idx=layer_idx,\n                )\n            if use_cache and not is_dynamic_cache:\n                next_past_key_values.append(next_cache)\n\n        hidden_states = self.final_norm(hidden_states)\n        logits = self.out_head(hidden_states.to(self.param_dtype))\n\n        loss = None\n        if labels is not None:\n            shift_logits = logits[..., :-1, :].contiguous()\n            shift_labels = labels[..., 1:].contiguous()\n            loss = nn.functional.cross_entropy(\n                shift_logits.view(-1, shift_logits.size(-1)),\n                shift_labels.view(-1),\n                ignore_index=-100,\n            )\n\n        if use_cache:\n            if is_dynamic_cache:\n                next_past_key_values = past_key_values\n            else:\n                next_past_key_values = tuple(next_past_key_values)\n        else:\n            next_past_key_values = None\n\n        if not return_dict:\n            outputs = (logits, next_past_key_values)\n            return ((loss,) + outputs) if loss is not None else outputs\n\n        return CausalLMOutputWithPast(\n            loss=loss,\n            logits=logits,\n            past_key_values=next_past_key_values,\n        )\n\n    def prepare_inputs_for_generation(\n        self,\n        input_ids,\n        past_key_values=None,\n        attention_mask=None,\n        **kwargs,\n    ):\n        has_past = past_key_values is not None\n        if has_past and hasattr(past_key_values, "get_seq_length"):\n            has_past = past_key_values.get_seq_length(0) > 0\n        elif has_past:\n            has_past = len(past_key_values) > 0\n\n        if has_past:\n            input_ids = input_ids[:, -1:]\n\n        return {\n            "input_ids": input_ids,\n            "attention_mask": attention_mask,\n            "past_key_values": past_key_values,\n            "use_cache": kwargs.get("use_cache", True),\n        }\n\n    @staticmethod\n    def _reorder_cache(past_key_values, beam_idx):\n        if past_key_values is None:\n            return past_key_values\n        if hasattr(past_key_values, "reorder_cache"):\n            past_key_values.reorder_cache(beam_idx)\n            return past_key_values\n\n        reordered = []\n        for layer_past in past_key_values:\n            reordered.append(\n                tuple(past_state.index_select(0, beam_idx) for past_state in layer_past)\n            )\n        return tuple(reordered)\n'}


In [4]:
assert CHECKPOINT.is_file(), f"Select an existing checkpoint: {CHECKPOINT}"
HELPER_DIR = Path.cwd() / "hf_export_helpers"
HELPER_DIR.mkdir(exist_ok=True)
for name, source in EXPORT_HELPERS.items():
    (HELPER_DIR / name).write_text(source, encoding="utf-8")


## Convert checkpoint
Uses the base tokenizer, matching the GRPO training script. Downloads the tokenizer if needed. Export output should be a dedicated folder containing only files you intend to upload.

In [5]:
subprocess.run([
    sys.executable, str(HELPER_DIR / "hf_export.py"),
    "--model_path", str(CHECKPOINT.resolve()),
    "--output_dir", str(EXPORT_DIR.resolve()),
    "--tokenizer_kind", "base",
], check=True)


Writing model shards: 100%|██████████| 1/1 [00:23<00:00, 23.79s/it]


Exported Hugging Face model folder to /home/jovyan/qwen3-grpo-hf


CompletedProcess(args=['/opt/conda/bin/python', '/home/jovyan/hf_export_helpers/hf_export.py', '--model_path', '/home/jovyan/grpo_training/checkpoints/rlvr_grpo_original_no_kl/qwen3-0.6B-rlvr-grpo-step00100.pth', '--output_dir', '/home/jovyan/qwen3-grpo-hf', '--tokenizer_kind', 'base'], returncode=0)

## Validate exported model
Loads it on CPU in a separate process and performs a short forward pass. This executes the custom architecture embedded above. Requires the reasoning_from_scratch package.

In [6]:
validation = """
import sys, torch
from transformers import AutoModelForCausalLM, AutoTokenizer
folder = sys.argv[1]
tokenizer = AutoTokenizer.from_pretrained(folder, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(folder, trust_remote_code=True).cpu().eval()
inputs = tokenizer("What is 1 + 1?", return_tensors="pt")
with torch.no_grad():
    output = model(**inputs)
assert torch.isfinite(output.logits).all(), "Non-finite output logits"
print("Export reload and forward pass succeeded:", tuple(output.logits.shape))
"""
subprocess.run([sys.executable, "-c", validation, str(EXPORT_DIR.resolve())], check=True)
EXPORT_VALIDATED = True


Loading weights: 100%|██████████| 313/313 [00:00<00:00, 5865.06it/s]
[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


Export reload and forward pass succeeded: (1, 8, 151936)


In [7]:
card = f"""---
language:
- en
license: apache-2.0
base_model: Qwen/Qwen3-0.6B-Base
tags:
- grpo
- math
- custom_code
---
# Qwen3 0.6B GRPO math fine-tune

Full-weight reinforcement learning from the Qwen3 0.6B base model using the reasoning-from-scratch implementation.

## Training
{TRAINING_NOTES}

Checkpoint: `{CHECKPOINT.name}`
Training dataset: https://huggingface.co/datasets/rasbt/math_full_minus_math500

## Evaluation
{EVALUATION_NOTES}

## Usage
Install `reasoning_from_scratch` and `transformers`. This export uses a custom architecture and requires `trust_remote_code=True`.

```python
from transformers import AutoTokenizer, AutoModelForCausalLM
model_id = "{REPO_ID}"
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(model_id, trust_remote_code=True)
```

Use the repository's math prompt template for comparable evaluation. Training focuses on math reasoning; no general assistant performance claim is made.
"""
(EXPORT_DIR / "README.md").write_text(card, encoding="utf-8")
print(card)
print("Files to upload:")
for path in sorted(EXPORT_DIR.iterdir()):
    print(path.name)


---
language:
- en
license: apache-2.0
base_model: Qwen/Qwen3-0.6B-Base
tags:
- grpo
- math
- custom_code
---
# Qwen3 0.6B GRPO math fine-tune

Full-weight reinforcement learning from the Qwen3 0.6B base model using the reasoning-from-scratch implementation.

## Training
GRPO without KL; 8 rollouts; maximum 1024 generated tokens; checkpoint step 100.

Checkpoint: `qwen3-0.6B-rlvr-grpo-step00100.pth`
Training dataset: https://huggingface.co/datasets/rasbt/math_full_minus_math500

## Evaluation
Step 100: 28/50 correct (56%) on the 50-example MATH-500 subset, with a 1024-token generation limit. Step 50: 21/50 (42%). These subset scores are not a full MATH-500 benchmark.

## Usage
Install `reasoning_from_scratch` and `transformers`. This export uses a custom architecture and requires `trust_remote_code=True`.

```python
from transformers import AutoTokenizer, AutoModelForCausalLM
model_id = "YOUR_USERNAME/qwen3-0.6b-grpo-math"
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remot

## Authenticate
Paste your token into the hidden input. Do not put it in a code cell. Authentication stays in memory for this session.

In [ ]:
from getpass import getpass
from huggingface_hub import HfApi

_token = "key"
api = HfApi(token=_token)
del _token
print("Authenticated as:", api.whoami()["name"])


Authenticated as: rristo


## Upload
Running this cell creates the destination model repository if needed and commits the converted model files. Existing repositories keep their existing visibility. Edit REPO_ID above first.

In [ ]:
assert EXPORT_VALIDATED, "Run conversion validation first."
REPO_ID = "rristo/qwen3-0.6b-grpo-math"
assert "/" in REPO_ID and "YOUR_USERNAME" not in REPO_ID, "Set REPO_ID first."
api.create_repo(repo_id=REPO_ID, repo_type="model", private=PRIVATE, exist_ok=True)
result = api.upload_folder(
    repo_id=REPO_ID,
    repo_type="model",
    folder_path=str(EXPORT_DIR),
    commit_message=f"Upload converted GRPO checkpoint {CHECKPOINT.name}",
    allow_patterns=["*.json", "*.safetensors", "*.py", "*.md", "*.model", "*.txt"],
)
print("Uploaded:", result.commit_url)
print("Model:", f"https://huggingface.co/{REPO_ID}")
